# ConvNeXt — A ConvNet for the 2020s (2022)

_Papers · Computer Vision_

**Take a plain ResNet and, one design change at a time, copy what made vision transformers win — until a pure ConvNet matches them.**

---

This notebook is a practice scaffold for the **ConvNeXt — A ConvNet for the 2020s (2022)** lesson. We build a tiny ConvNeXt block by hand — depthwise conv for cheap spatial mixing, an inverted-bottleneck MLP for the channel mixing — train it on a slice of MNIST, then ablate the wide middle to see what it is worth. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Count where a ConvNeXt block spends its parameters

Before building the block, tally the weights for one block at width `C=32`, `7x7` kernel, expansion `r=4`. The depthwise conv uses one `7x7` filter per channel, so it scales with `C`. The two `1x1` convs are dense channel mixers, so they scale with `C^2` — and they hold the bulk of the parameters, just like a transformer's MLP dwarfs its attention.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

torch.manual_seed(0)

# Parameter counts for one block: C=32, kernel 7x7, expansion r=4.
C = 32
K = 7
r = 4
dw      = C * K * K          # 32 * 49 = 1568  depthwise weights (one 7x7 filter per channel)
expand  = C * (r * C)        # 32 * 128 = 4096  1x1 expand  (C -> 4C)
project = (r * C) * C        # 128 * 32 = 4096  1x1 project (4C -> C)
print("depthwise 7x7 weights :", dw)                       # 1568
print("1x1 expand  (C->4C)   :", expand)                   # 4096
print("1x1 project (4C->C)   :", project)                  # 4096
print("channel-mix / spatial :", (expand + project), "vs", dw)   # 8192 vs 1568

### Step 2 — Define the ConvNeXt block and the tiny network

The block (Section 2.6, Figure 4) follows the transformer-inspired recipe: a depthwise `7x7` conv for spatial mixing, **one** LayerNorm, then a per-pixel MLP — two `1x1` convs (written as `Linear` over channels-last tensors) with **one** GELU and an `r`-times inverted bottleneck — wrapped in a residual. `TinyConvNeXt` stacks a few blocks behind a small patchify stem and a global-average-pool head.

In [ ]:
# The ConvNeXt block (Section 2.6, Figure 4). r toggles the inverted-bottleneck ablation.
class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, r=4):
        super().__init__()
        # (a) depthwise 7x7 conv: groups=dim => one filter per channel (spatial mixing only).
        #     padding=3 keeps HxW.
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        # (b) ONE LayerNorm over channels.
        self.norm = nn.LayerNorm(dim)
        # (c) 1x1 expand  C -> rC   (the inverted bottleneck).
        self.pw1 = nn.Linear(dim, r * dim)
        # (d) ONE GELU.
        self.act = nn.GELU()
        # (e) 1x1 project rC -> C.
        self.pw2 = nn.Linear(r * dim, dim)

    def forward(self, x):                       # x: (N, C, H, W)
        h = self.dwconv(x)                      # depthwise spatial mixing
        h = h.permute(0, 2, 3, 1)               # (N, H, W, C): channels last for LayerNorm + Linear
        h = self.norm(h)                        # one LayerNorm
        h = self.pw1(h)                         # 1x1 expand
        h = self.act(h)                         # GELU
        h = self.pw2(h)                         # 1x1 project
        h = h.permute(0, 3, 1, 2)               # back to (N, C, H, W)
        return x + h                            # residual (ResNet idea)

class TinyConvNeXt(nn.Module):
    def __init__(self, dim=32, blocks=3, r=4, classes=10):
        super().__init__()
        self.stem = nn.Conv2d(1, dim, kernel_size=2, stride=2)   # tiny "patchify" stem (28x28 -> 14x14)
        self.blocks = nn.Sequential(*[ConvNeXtBlock(dim, r=r) for _ in range(blocks)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = x.mean(dim=(2, 3))                  # global average pool -> (N, C)
        x = self.norm(x)
        return self.head(x)

### Step 3 — Load a small MNIST subset and helpers

To keep this fast on CPU we use a 3,000-image training subset and a 1,000-image test subset. `evaluate` reports test accuracy, and `train` builds a fresh `TinyConvNeXt` for a given expansion ratio `r` and trains it with **AdamW** — the modern optimizer recipe the paper adopts (Section 2.1).

In [ ]:
# A small MNIST subset (fast on CPU).
tf = transforms.ToTensor()
train_full = datasets.MNIST(root="./data", train=True,  download=True, transform=tf)
test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=tf)
train_set = torch.utils.data.Subset(train_full, range(3000))
test_set  = torch.utils.data.Subset(test_full,  range(1000))
train_dl = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
test_dl  = torch.utils.data.DataLoader(test_set,  batch_size=256)

def evaluate(net):
    net.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_dl:
            preds = net(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total

def train(r, epochs=6, lr=2e-3):
    torch.manual_seed(0)
    net = TinyConvNeXt(r=r)
    opt = torch.optim.AdamW(net.parameters(), lr=lr)   # AdamW: the modern recipe (Section 2.1)
    for ep in range(epochs):
        net.train()
        for x, y in train_dl:
            loss = F.cross_entropy(net(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        print(f"  epoch {ep}  test-acc {evaluate(net):.3f}")
    return evaluate(net)

### Step 4 — Train with the inverted bottleneck, then ablate it

Train once with the full `4x` inverted bottleneck (`r=4`), then again with the wide middle collapsed (`r=1`) so the two `1x1` convs keep the width flat. Everything else — depth, kernel, norm, optimizer, data, seed — is identical, so the accuracy gap isolates what the inverted bottleneck's capacity is worth.

In [ ]:
print("\nWITH inverted bottleneck (r=4):")
acc_r4 = train(r=4)
print("WITHOUT wide middle (ABLATION, r=1):")
acc_r1 = train(r=1)
print(f"\nfinal test accuracy  r=4: {acc_r4:.3f}   r=1: {acc_r1:.3f}")
# r=4 reaches ~0.94 on this tiny subset; r=1 drops to ~0.88 -- the inverted bottleneck's capacity is gone.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_On a small MNIST subset, does the hand-built ConvNeXt stack learn, and does collapsing the inverted bottleneck (expansion r=4 -> r=1, the ablation) lower accuracy?_

### Step 1 — Rebuild the network for a self-contained run

This panel reconstructs the block and network on its own seed so the figure stands alone. The block is the same as before, written compactly; `run` trains a `TinyConvNeXt` for a given `r` and returns the **per-epoch** test accuracy so we can plot the learning curve.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

torch.manual_seed(0)

class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, r=4):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)  # depthwise
        self.norm = nn.LayerNorm(dim)
        self.pw1 = nn.Linear(dim, r * dim)
        self.act = nn.GELU()
        self.pw2 = nn.Linear(r * dim, dim)

    def forward(self, x):
        h = self.dwconv(x).permute(0, 2, 3, 1)          # channels last
        h = self.norm(h)                                # LayerNorm
        h = self.pw1(h)                                 # 1x1 expand
        h = self.act(h)                                 # GELU
        h = self.pw2(h)                                 # 1x1 project
        return x + h.permute(0, 3, 1, 2)                # residual

class TinyConvNeXt(nn.Module):
    def __init__(self, dim=32, blocks=3, r=4, classes=10):
        super().__init__()
        self.stem = nn.Conv2d(1, dim, kernel_size=2, stride=2)
        self.blocks = nn.Sequential(*[ConvNeXtBlock(dim, r=r) for _ in range(blocks)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = x.mean(dim=(2, 3))                          # global average pool
        return self.head(self.norm(x))

tf = transforms.ToTensor()
tr = torch.utils.data.Subset(datasets.MNIST("./data", True,  download=True, transform=tf), range(3000))
te = torch.utils.data.Subset(datasets.MNIST("./data", False, download=True, transform=tf), range(1000))
tr_dl = torch.utils.data.DataLoader(tr, batch_size=128, shuffle=True)
te_dl = torch.utils.data.DataLoader(te, batch_size=256)

def acc(net):
    net.eval()
    c = 0
    t = 0
    with torch.no_grad():
        for x, y in te_dl:
            c += (net(x).argmax(1) == y).sum().item()
            t += y.numel()
    return c / t

def run(r, epochs=6):
    torch.manual_seed(0)
    net = TinyConvNeXt(r=r)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3)
    hist = []
    for _ in range(epochs):
        net.train()
        for x, y in tr_dl:
            loss = F.cross_entropy(net(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        hist.append(round(acc(net), 3))
    return hist

### Step 2 — Run both settings and compare the curves

Train `r=4` and `r=1` and print their per-epoch accuracy histories. The full inverted bottleneck climbs to roughly `0.94`, while the collapsed version plateaus near `0.88` — the same gap the reference run showed, now visible epoch by epoch.

In [ ]:
print("r=4:", run(4))
print("r=1:", run(1))
# r=4 -> climbs to ~0.94. r=1 -> plateaus ~0.88 (inverted bottleneck collapsed).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your tiny ConvNeXt classifies the MNIST subset with the $4\times$ inverted
            bottleneck. Set the expansion ratio $r$ from $4$ to $1$ (the two $1\times1$ convs now keep the width
            flat instead of widening to $4\times$ and back) and retrain. What happens to accuracy, and what does
            that show about the inverted bottleneck?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the middle width: in the block, replace hidden = 4 * dim with hidden = 1 * dim; keep depth, kernel, norm, GELU, optimizer, data, and seed identical. — _An honest ablation changes exactly one thing &mdash; the expansion ratio &mdash; so any difference is attributable to it._
- Retrain and compare test accuracy: with $r=4$ it is higher; with $r=1$ it drops (in our run ~0.94 to ~0.88). — _The wide middle gives the per-pixel MLP capacity to transform features; collapsing it to width $C$ removes most of the block's channel-mixing power._
- Conclude the inverted bottleneck (not just "having two 1x1 convs") supplies real capacity; the gap is what the $4\times$ expansion is worth. — _Both runs share architecture, depth, and parameter shape except the hidden width, isolating the expansion as the cause._

**Answer:** With $r=1$ the wide middle is gone and test accuracy drops (in our run ~0.94 &rarr; ~0.88). It
                 does not collapse to chance &mdash; the depthwise conv and residual still mix and carry signal &mdash;
                 but the block loses the channel-mixing capacity that the $4\times$ expansion provides, exactly as a
                 transformer MLP shrunk to no hidden expansion would weaken. Since the two runs are identical except
                 for the expansion ratio, this isolates the inverted bottleneck as a real source of capacity. The
                 CODEVIZ panel shows the gap.

</details>

**Problem 2.** Count the parameters of one ConvNeXt block at channel width $C=64$, kernel $7\times7$, expansion
            $r=4$ (ignore biases and LayerNorm). Where do most parameters live, and what does that tell you about
            the block's design?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Depthwise $7\times7$ conv: $C \times 7 \times 7 = 64 \times 49 = 3{,}136$ weights (one $7\times7$ filter per channel). — _Depthwise means one filter per channel, so it scales with $C$, not $C^2$ &mdash; cheap spatial mixing._
- Expand $1\times1$: $C \times rC = 64 \times 256 = 16{,}384$. Project $1\times1$: $rC \times C = 256 \times 64 = 16{,}384$. Total $1\times1$: $32{,}768$. — _The $1\times1$ convs are dense channel mixers, so they scale like $C^2$ &mdash; the bulk of the cost._
- Compare: $32{,}768$ in channel mixing vs $3{,}136$ in spatial mixing &mdash; about $90\%$ of the weights are in the $1\times1$ MLP. — _This mirrors a transformer block, whose MLP holds most parameters while attention/spatial-mixing is comparatively light._

**Answer:** Depthwise conv: $64\times49 = 3{,}136$. The two $1\times1$ convs: $64\times256 + 256\times64 =
                 32{,}768$. So ~$91\%$ of the block's weights are in the channel-mixing $1\times1$ MLP, and only
                 ~$9\%$ in the spatial depthwise conv. The design deliberately makes spatial mixing cheap (depthwise,
                 scales with $C$) and spends its parameters on the wide channel MLP (scales with $C^2$) &mdash;
                 exactly the budget split of a transformer block.

</details>

**Problem 3.** The paper starts at ResNet-50 ($76.1\%$) and, before changing any architecture, just trains it
            with the modern recipe (AdamW, augmentation, regularization) to $78.8\%$ (&sect;2.1). Why does the
            paper insist on doing this first, and what does it say about the "transformers beat CNNs" claim?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the $+2.7\%$ comes purely from the training recipe, with the identical ResNet-50 architecture. — _It isolates recipe from architecture &mdash; the same network, trained the modern way, is already much better._
- Recognize that early "ViT/Swin beats ResNet" comparisons used the modern recipe for the transformer but an old recipe for the ResNet. — _That is an unfair comparison: part of the measured gap was the recipe, not the architecture._
- Conclude you must control the training recipe before attributing a gap to attention; only then can the roadmap fairly credit each design change. — _Controlled comparison is the paper's core method &mdash; change one thing at a time, recipe included._

**Answer:** Training the same ResNet-50 with the transformer-era recipe lifts it $76.1\%\to 78.8\%$ with no
                 architecture change, so a large slice of the apparent "transformer advantage" was really the
                 recipe (AdamW, Mixup/CutMix/RandAugment, stochastic depth, ~300 epochs), not attention. The
                 paper insists on fixing the recipe first so that every later roadmap step is a fair, one-variable
                 comparison. The takeaway: "transformers beat CNNs" was partly an apples-to-oranges claim, and once
                 controlled, a pure ConvNet (ConvNeXt-T, $82.1\%$) edges out Swin-T ($81.3\%$).

</details>